# Silver — ERP Customer Demographics
Birth date and gender per customer from the ERP.

`bronze.erp_cust_az12` → `silver.erp_customers`

## Init

In [ ]:
import os, sys
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DateType
from pyspark.sql.window import Window

# Make src/ importable from wherever this notebook runs (git folder, bundle, VS Code sync)
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from lakehouse.transforms import trim_strings, normalize, rename, yyyymmdd_to_date

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.erp_cust_az12")

## Transformations

### Trim all string columns

In [ ]:
df = trim_strings(df)

### Clean customer id
Some ids carry a `NAS` prefix (`NASAW00011000`). Strip it so the id matches `crm_customers.customer_number` (`AW00011000`).

In [ ]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"), F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

### Invalidate future birth dates

In [ ]:
df = df.withColumn("bdate", F.when(col("bdate") > F.current_date(), None).otherwise(col("bdate")))

### Normalize gender
The ERP mixes codes and words (`F`, `FEMALE`, `M`, `MALE`).

In [ ]:
df = normalize(df, "gen", {"F": "Female", "FEMALE": "Female", "M": "Male", "MALE": "Male"})

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
df = rename(df, RENAME_MAP)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.erp_customers")

In [ ]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10;